# Day 4

## Tokenizing with code

In [2]:
!pip install tiktoken

   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/879.1 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 879.1/879.1 kB 4.8 MB/s eta 0:00:00


In [3]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [4]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [5]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [6]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [9]:
!pip install ollama

In [12]:
import os
from dotenv import load_dotenv
import ollama

load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

# if not api_key:
#     print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# elif not api_key.startswith("sk-proj-"):
#     print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# else:
#     print("API key found and looks good so far!")

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [16]:
from openai import OpenAI

openai = OpenAI(base_url=ollama_base_url,api_key="ollama")

### A message to OpenAI is a list of dicts

In [17]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [23]:
!ollama list

NAME                                   ID              SIZE      MODIFIED    
deepseek-r1:1.5b                       e0979632db5a    1.1 GB    6 hours ago    
llama3.2:1b                            baf6a787fdff    1.3 GB    7 hours ago    
nomic-embed-text:latest                0a109f422b47    274 MB    2 weeks ago    
artifish/llama3.2-uncensored:latest    c73bea26e004    2.2 GB    2 weeks ago    


In [24]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"Hello Ed! How's it going? Is there anything on your mind that you'd like to chat about, or are you just looking for some friendly conversation?"

### OK let's now ask a follow-up question

In [25]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [26]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

"I don't have any information that suggests I know your name. I'm a large language model, I don't have the ability to store or access personal information about individual users. Each time you interact with me, it's a new conversation and I don't retain anything from previous conversations. Is there anything else I can help you with?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [27]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [28]:
response = openai.chat.completions.create(model="llama3.2:1b", messages=messages)
response.choices[0].message.content

'Your name is Ed, right?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

